# Introduction to AI Agents
### Practice Notebook

**Assumed pre-installed libraries:** none required (pure Python standard library)

This notebook has no live LLM API calls wired in. Instead, we simulate the
"reasoning" step of an agent with simple rule-based stand-ins, so you can see
the *structure* of the agent loop clearly before layering a real LLM on top
of it. Wherever you see `# TODO: replace with a real LLM call`, that's where
a production system would call Claude (or another model) instead.


In [1]:
import os
from huggingface_hub import InferenceClient

HF_TOKEN = os.getenv("HF_TOKEN")

## 1. A simple LLM call vs. an agent

A single LLM call is:

```text
User → LLM → Answer
```

An agent is a loop:

```text
Goal → LLM → Tool → Observation → LLM → Tool → Observation → Answer
```

The important difference is **interaction with an external environment**. The agent can obtain information that was not present in the original prompt.

In [2]:
def single_llm_call(user_request: str) -> str:
    """A single LLM call: one input, one output, done. No ability to check
    its own work or gather more information before answering.
    """
    # TODO: replace with a real LLM call
    return f"(stub) Here's my best attempt at handling: '{user_request}'"

print(single_llm_call("What's the cheapest flight from Bengaluru to Mumbai next Friday?"))


(stub) Here's my best attempt at handling: 'What's the cheapest flight from Bengaluru to Mumbai next Friday?'


Notice the single call above can only ever *guess* -- it has no way to
actually check flight prices. This is exactly the gap the agent loop below
is designed to close.


## 2. The agent loop: Perceive → Reason → Act

```
        +-------------------------------------+
        |                                     |
        v                                     |
   [PERCEIVE]  ---->  [REASON]  ---->  [ACT]  -+
   (read current       (LLM decides    (call a tool,
    state / last          what to do    take an action)
    tool result)           next)
```

Let's build this loop for a tiny toy task: an agent that needs to find the
cheapest of several flight options. "Perceive" starts as the user's request;
each "act" step calls a (fake) tool; the tool's result becomes the next
thing to "perceive".


In [3]:
# Deterministic fake flight data.
# Deterministic data makes the notebook easier to reproduce.

FLIGHTS = [
    {"airline": "IndiGo", "price": 4200},
    {"airline": "Air India", "price": 5100},
    {"airline": "Vistara", "price": 3900},
    {"airline": "SpiceJet", "price": 4500},
]

# Seat availability for Exercise 1.1
SEAT_AVAILABLE = {
    "Vistara": False,
    "IndiGo": True,
    "SpiceJet": True,
    "Air India": True,
}


def search_flights_tool(origin: str, destination: str) -> list:
    """Fake flight-search API."""
    return FLIGHTS.copy()


def check_seat_availability(airline: str, price: int) -> bool:
    """Fake seat-availability API."""
    return SEAT_AVAILABLE.get(airline, False)


def sort_flights_by_price(flight_options: list) -> list:
    return sorted(flight_options, key=lambda x: x["price"])

In [4]:
def reason_pick_cheapest(flight_options: list) -> tuple:
    """The REASON step. In a real agent this would be an LLM call deciding
    what to do with the observed tool result; here we hard-code the logic
    ("pick the minimum price") so the loop structure is easy to see.
    # TODO: replace with a real LLM call that reasons over `flight_options`
    """
    return min(flight_options, key=lambda x: x[1])

In [6]:
def reason_pick_cheapest(flight_options):
    cheapest = min(flight_options, key=lambda x: x["price"])
    return cheapest["airline"], cheapest["price"]


def run_agent(origin: str, destination: str):
    print(f"[PERCEIVE] Goal: find cheapest flight {origin} -> {destination}")

    print("[ACT] Calling search_flights_tool(...)")
    flight_options = search_flights_tool(origin, destination)

    print(f"[PERCEIVE] Observed tool result: {flight_options}")

    print("[REASON] Deciding which option is best...")
    best = reason_pick_cheapest(flight_options)

    print(f"[ACT] Final answer: book {best[0]} at Rs.{best[1]}")

    return best


run_agent("Bengaluru", "Mumbai")


[PERCEIVE] Goal: find cheapest flight Bengaluru -> Mumbai
[ACT] Calling search_flights_tool(...)
[PERCEIVE] Observed tool result: [{'airline': 'IndiGo', 'price': 4200}, {'airline': 'Air India', 'price': 5100}, {'airline': 'Vistara', 'price': 3900}, {'airline': 'SpiceJet', 'price': 4500}]
[REASON] Deciding which option is best...
[ACT] Final answer: book Vistara at Rs.3900


('Vistara', 3900)

**Exercise 1.1:** The loop above only runs one perceive-reason-act cycle. Add
a second cycle: after picking the cheapest flight, call a second fake tool
`check_seat_availability(airline, price)` (write it yourself -- have it
randomly return True/False), and if it returns `False`, go back and pick the
*next* cheapest option instead. This is the essence of a *multi-step* agent
loop instead of a single pass.


## 3. The ReAct pattern: interleaving reasoning and acting

ReAct (Yao et al., 2022) interleaves explicit reasoning traces with actions:
the model writes out *why* it's about to do something, then does it, then
reasons about the result, then acts again. Let's make the reasoning trace
visible as text, the way a real ReAct-style LLM transcript would look.


In [7]:
def react_agent_transcript(origin: str, destination: str):
    transcript = []

    transcript.append(f"Thought: I need to find flights from {origin} to {destination}. "
                       f"I should call the flight search tool.")
    transcript.append("Action: search_flights_tool(origin, destination)")
    flights = search_flights_tool(origin, destination)
    transcript.append(f"Observation: {flights}")

    transcript.append("Thought: Now I should compare prices and pick the cheapest one.")
    best = reason_pick_cheapest(flights)
    transcript.append(f"Action: select_flight({best[0]})")
    transcript.append(f"Observation: {best[0]} confirmed available at Rs.{best[1]}")

    transcript.append(f"Thought: I have enough information to answer the user now.")
    transcript.append(f"Final Answer: Book {best[0]} for Rs.{best[1]}, the cheapest option found.")

    return transcript

for line in react_agent_transcript("Bengaluru", "Mumbai"):
    print(line)


Thought: I need to find flights from Bengaluru to Mumbai. I should call the flight search tool.
Action: search_flights_tool(origin, destination)
Observation: [{'airline': 'IndiGo', 'price': 4200}, {'airline': 'Air India', 'price': 5100}, {'airline': 'Vistara', 'price': 3900}, {'airline': 'SpiceJet', 'price': 4500}]
Thought: Now I should compare prices and pick the cheapest one.
Action: select_flight(Vistara)
Observation: Vistara confirmed available at Rs.3900
Thought: I have enough information to answer the user now.
Final Answer: Book Vistara for Rs.3900, the cheapest option found.


**Exercise 1.2:** Notice every `Thought:` line above is hard-coded text, not
actually generated reasoning -- the *agent* isn't really "deciding" anything,
we are. Identify: which lines would change if the tool returned a different
result (e.g., an empty flight list, meaning no flights are available)? Add a
branch to `react_agent_transcript` that produces a sensible transcript for
that case.

**Exercise 1.3:** Compare `run_agent` (Part 2) and `react_agent_transcript`
(Part 3) -- what's actually different about the two implementations,
structurally? (Hint: it's not the underlying logic, which is the same --
it's what's made *visible* and *inspectable*.) Why might that visibility
matter when debugging a real agent that misbehaves?


## 4. When *not* to use an agent

Agents add latency (multiple LLM calls instead of one), cost, and
unpredictability. Not every task needs a loop.

**Exercise 1.4 (discussion, no code):** For each task below, decide whether
it's better solved with (a) a single LLM call, (b) a fixed multi-step
workflow (Day 3 topic, predefined steps, no loop), or (c) a full
perceive-reason-act agent loop -- and justify your answer in one sentence:

1. Translate a sentence from English to Hindi.
2. Answer "What's on my calendar tomorrow?" by checking a calendar API.
3. "Plan and book my entire 5-day trip to Goa, adjusting for weather and
   budget as you go, and re-book anything that falls through."
4. Classify a support ticket as "billing," "technical," or "other."


For translating a sentence from English to Hindi, a **single LLM call (a)** is best because it is a simple, self-contained task that does not require multiple steps or decision-making. For answering “What's on my calendar tomorrow?” using a calendar API, a **fixed multi-step workflow (b)** is more suitable because the steps are predictable: determine tomorrow's date, call the calendar API, retrieve the events, and present them to the user. For planning and booking an entire 5-day trip to Goa while continuously adjusting for weather, budget, and failed bookings, a **full perceive-reason-act agent loop (c)** is the best choice because the system needs to observe changing conditions, reason about alternatives, take actions, and adapt when circumstances change. Finally, classifying a support ticket as “billing,” “technical,” or “other” is best handled with a **single LLM call (a)** because it is a straightforward classification task that can be completed in one step.
